# Customizing imports

This notebook writes the files its exercises import (an `exports.py` module, a `plugins/` folder and a `reports/` package) into the current directory itself: run the next cell first. The exercises follow the order of the course: importing by a name known at run time, from a file, on first access, then stepping into the import machinery itself.

In [ ]:
from pathlib import Path

Path("plugins").mkdir(exist_ok=True)
Path("reports").mkdir(exist_ok=True)

ROWS = [
  {"product": "pen", "price": 1.5},
  {"product": "notebook", "price": 3.2},
]

## Loading a class named in a configuration

An export application reads the class to use from its configuration, in the form `"module:Class"`, like the entry points of a `pyproject.toml`. The next cell writes the `exports.py` module, which defines two of them.

Write a function `load_object(reference)` that imports the module named before the `:` and returns the object named after it. A reference whose module or object doesn't exist raises a `ValueError` that names it.

With this code:

```python
for reference in ["exports:ExportCsv", "exports:ExportJson"]:
  print(load_object(reference)().export(ROWS))

for reference in ["exports:ExportPdf", "exports_pdf:ExportPdf"]:
  try:
    load_object(reference)
  except ValueError as error:
    print(error)
```

You should get:

```text
product,price
pen,1.5
notebook,3.2
[{"product": "pen", "price": 1.5}, {"product": "notebook", "price": 3.2}]
unknown reference: exports:ExportPdf
unknown reference: exports_pdf:ExportPdf
```

**Useful documentation**

- [`str.partition`](https://docs.python.org/3/library/stdtypes.html#str.partition)

In [ ]:
%%writefile exports.py
import json


class ExportCsv:
  def export(self, rows):
    columns = list(rows[0])
    result = [",".join(columns)]
    result += [",".join(str(row[c]) for c in columns) for row in rows]
    return "\n".join(result)


class ExportJson:
  def export(self, rows):
    return json.dumps(rows, ensure_ascii=False)

In [ ]:
# Your code here

### Additional hints

`"exports:ExportCsv".partition(":")` is `("exports", ":", "ExportCsv")`. `importlib.import_module("exports")` returns the module (the one in `sys.modules` if it's already imported), and `getattr(module, "ExportCsv")` the class. A module that can't be found raises `ModuleNotFoundError`, a name missing from the module `AttributeError`: catch both, and raise the `ValueError` with `from error` to keep the cause in the traceback.

```python
import importlib


def load_object(reference):
  module_name, _, object_name = reference.partition(":")
  try:
    module = ...
    return ...
  except (ModuleNotFoundError, AttributeError) as error:
    raise ValueError(f"unknown reference: {reference}") from error
```

### Solution

In [ ]:
import importlib


def load_object(reference):
  module_name, _, object_name = reference.partition(":")
  try:
    module = importlib.import_module(module_name)
    return getattr(module, object_name)
  except (ModuleNotFoundError, AttributeError) as error:
    raise ValueError(f"unknown reference: {reference}") from error


for reference in ["exports:ExportCsv", "exports:ExportJson"]:
  print(load_object(reference)().export(ROWS))

for reference in ["exports:ExportPdf", "exports_pdf:ExportPdf"]:
  try:
    load_object(reference)
  except ValueError as error:
    print(error)

`ModuleNotFoundError` also comes from an import *inside* the loaded module: if `exports.py` imported a library that isn't installed, this solution would say "unknown reference" for a module that exists. Its `name` attribute says which module is missing: only turn the error into a `ValueError` if `error.name == module_name`, and let the others propagate.

## Loading the plugins of a folder

The export plugins are now files placed in a `plugins/` folder, outside `sys.path`. Each defines an `EXTENSION` constant and an `export(rows)` function. The next two cells write two of them, `markdown.py` and `html.py`.

1. Write a function `load_plugin(path)` that loads the file `path` (a `Path`) as a module, under the name `plugin_<file name without extension>`, and returns it.
2. Write a function `load_plugins(folder)` that returns a dictionary of the plugins of all the `.py` files of `folder`, keyed by the file name without extension.

With this code:

```python
plugins = load_plugins(Path("plugins"))
print(sorted(plugins))
print(plugins["markdown"].export(ROWS))
print(plugins["html"].EXTENSION, plugins["html"].__name__)
```

You should get:

```text
['html', 'markdown']
| product | price |
|---|---|
| pen | 1.5 |
| notebook | 3.2 |
.html plugin_html
```

**Useful documentation**

- [`pathlib.Path.glob`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob)

In [ ]:
%%writefile plugins/markdown.py
EXTENSION = ".md"


def export(rows):
  columns = list(rows[0])
  header = "| " + " | ".join(columns) + " |"
  separator = "|" + "---|" * len(columns)
  body = [
    "| " + " | ".join(str(row[c]) for c in columns) + " |" for row in rows
  ]
  return "\n".join([header, separator, *body])

In [ ]:
%%writefile plugins/html.py
EXTENSION = ".html"


def export(rows):
  table_rows = [
    "<tr>" + "".join(f"<td>{value}</td>" for value in row.values()) + "</tr>"
    for row in rows
  ]
  return "<table>\n" + "\n".join(table_rows) + "\n</table>"

In [ ]:
# Your code here

### Additional hints

Three steps: `importlib.util.spec_from_file_location(name, path)` returns the file's *spec* (its module name, its origin, its loader); `importlib.util.module_from_spec(spec)` creates an empty module; `spec.loader.exec_module(module)` runs the file's code in that module. Register the module in `sys.modules` before running it, as `import` does: plugin code that looks up its own module (`dataclasses`, `pickle`) finds it. `path.stem` is `"markdown"` for `plugins/markdown.py`, and `folder.glob("*.py")` returns the files in no particular order.

```python
import importlib.util
import sys


def load_plugin(path):
  name = f"plugin_{path.stem}"
  spec = ...
  module = ...
  sys.modules[name] = module
  ...
  return module


def load_plugins(folder):
  return {path.stem: ... for path in sorted(folder.glob("*.py"))}
```

### Solution

In [ ]:
import importlib.util
import sys


def load_plugin(path):
  name = f"plugin_{path.stem}"
  spec = importlib.util.spec_from_file_location(name, path)
  module = importlib.util.module_from_spec(spec)
  sys.modules[name] = module
  spec.loader.exec_module(module)
  return module


def load_plugins(folder):
  return {path.stem: load_plugin(path) for path in sorted(folder.glob("*.py"))}


plugins = load_plugins(Path("plugins"))
print(sorted(plugins))
print(plugins["markdown"].export(ROWS))
print(plugins["html"].EXTENSION, plugins["html"].__name__)

Why not `sys.path.insert(0, "plugins")` then `import_module("html")`? The `html.py` plugin has the name of a standard library module, which Jupyter has already imported: `import_module` would return that one, without even reading the file. And if it weren't imported yet, the plugin would shadow the standard module for the whole program. Loading the file by its path, under a prefixed name, avoids both. A plugin whose code raises during `exec_module` stays registered in `sys.modules`, half initialized: an application removes its name (`sys.modules.pop(name)`) in an `except` before re-raising the error.

## A submodule imported on first access

The `reports` package has two submodules, `pdf` and `spreadsheet`, which the next two cells write: each prints a message when it's imported. We want `import reports` to import neither of them, and `reports.pdf` to import `reports.pdf` on first access only.

1. Write the `reports/__init__.py` file, with `%%writefile reports/__init__.py` as the first line of your cell: a list `__all__` of the submodules, and a module `__getattr__(name)` function that imports the requested submodule, or raises Python's usual `AttributeError` for any other name.
2. Add a module `__dir__()` function, so that `dir(reports)` also lists the submodules not imported yet.

To check, run this code in a new cell: its first lines remove the package and its submodules from `sys.modules`, so that the cell can be run again after a change to your file.

```python
import importlib
import sys

for name in [n for n in sys.modules if n == "reports" or n.startswith("reports.")]:
  del sys.modules[name]
importlib.invalidate_caches()

import reports

print("pdf" in vars(reports), "spreadsheet" in dir(reports))
print(reports.pdf.generate())
print(reports.pdf.generate())
print("pdf" in vars(reports))
try:
  reports.excel
except AttributeError as error:
  print(error)
```

You should get:

```text
False True
loading reports.pdf
PDF report
PDF report
True
module 'reports' has no attribute 'excel'
```

**Useful documentation**

- [Customizing module attribute access](https://docs.python.org/3/reference/datamodel.html#customizing-module-attribute-access)

In [ ]:
%%writefile reports/pdf.py
print(f"loading {__name__}")


def generate():
  return "PDF report"

In [ ]:
%%writefile reports/spreadsheet.py
print(f"loading {__name__}")


def generate():
  return "spreadsheet report"

In [ ]:
# Your code here

### Additional hints

Python only calls a module's `__getattr__` for a name missing from the module. `importlib.import_module(f".{name}", __name__)` imports the submodule relative to the package (`__name__` is `"reports"` in `__init__.py`), and sets it as an attribute of the package: on the second access, `__getattr__` is no longer called. The usual message is `f"module {__name__!r} has no attribute {name!r}"`. `__dir__()` takes no argument: `globals()` gives the names already defined in the module.

```python
%%writefile reports/__init__.py
import importlib

__all__ = ["pdf", "spreadsheet"]


def __getattr__(name):
  if name in __all__:
    return ...
  raise AttributeError(...)


def __dir__():
  return sorted(...)
```

### Solution

In [ ]:
%%writefile reports/__init__.py
import importlib

__all__ = ["pdf", "spreadsheet"]


def __getattr__(name):
  if name in __all__:
    return importlib.import_module(f".{name}", __name__)
  raise AttributeError(f"module {__name__!r} has no attribute {name!r}")


def __dir__():
  return sorted(set(globals()) | set(__all__))

In [ ]:
import importlib
import sys

for name in [n for n in sys.modules if n == "reports" or n.startswith("reports.")]:
  del sys.modules[name]
importlib.invalidate_caches()

import reports

print("pdf" in vars(reports), "spreadsheet" in dir(reports))
print(reports.pdf.generate())
print(reports.pdf.generate())
print("pdf" in vars(reports))
try:
  reports.excel
except AttributeError as error:
  print(error)

`from reports import spreadsheet` goes through `__getattr__` too: `from` reads the package's attribute, then, if it's missing, tries to import the submodule. On the other hand, `from reports import *` imports every name of `__all__`, and so both submodules: the laziness is gone.

## An import hook

Two finders added to `sys.meta_path`: the first serves modules that exist on no disk, the second (optional) refuses some imports.

### Modules served from a dictionary

Write a class `DictFinder`, both a finder (`importlib.abc.MetaPathFinder`) and a loader (`importlib.abc.Loader`), which receives a dictionary `sources` (module name → source code) when it's created and serves these modules to `import`. An instance of the class, inserted at the head of `sys.meta_path`, must make the modules of `SOURCES`, defined in the next cell, importable. `exec_module` prints `executing <module name>` before running the code.

The cell must be runnable again: before inserting your finder, remove those of a previous run from `sys.meta_path`, and `units` from `sys.modules`.

With this code:

```python
import units
import units

print(units.to_cm(10))
```

You should get:

```text
executing units
25.4
```

In [ ]:
SOURCES = {
  "units": """
CM_PER_INCH = 2.54


def to_cm(inches):
  return inches * CM_PER_INCH
""",
}

In [ ]:
# Your code here

#### Additional hints

`import units` queries the finders of `sys.meta_path` in turn, through their `find_spec(name, path, target)`: yours returns `importlib.util.spec_from_loader(name, self)` for a name in `sources`, and `None` otherwise, to hand over to the next finder. The import machinery then creates the module, registers it in `sys.modules` and calls `exec_module(module)`, which runs the source code with `exec(code, module.__dict__)`. The second `import units` finds the module in `sys.modules`: neither `find_spec` nor `exec_module` is called, hence a single message. A class redefined on each run of the cell is no longer the same class: remove the old finders by their class name, `type(finder).__name__`.

```python
import importlib.abc
import importlib.util
import sys


class DictFinder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
  def __init__(self, sources):
    self.sources = sources

  def find_spec(self, name, path, target=None):
    if name not in self.sources:
      return ...
    return ...

  def exec_module(self, module):
    print(f"executing {module.__name__}")
    ...


sys.meta_path[:] = [
  finder for finder in sys.meta_path if type(finder).__name__ != "DictFinder"
]
sys.modules.pop("units", None)
sys.meta_path.insert(0, ...)
```

#### Solution

In [ ]:
import importlib.abc
import importlib.util
import sys


class DictFinder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
  def __init__(self, sources):
    self.sources = sources

  def find_spec(self, name, path, target=None):
    if name not in self.sources:
      return None  # to the next finder of sys.meta_path
    return importlib.util.spec_from_loader(name, self)

  def exec_module(self, module):
    print(f"executing {module.__name__}")
    exec(self.sources[module.__name__], module.__dict__)


# Finders and module of a previous run of the cell
sys.meta_path[:] = [
  finder for finder in sys.meta_path if type(finder).__name__ != "DictFinder"
]
sys.modules.pop("units", None)
sys.meta_path.insert(0, DictFinder(SOURCES))

import units
import units

print(units.to_cm(10))

### (Optional) Refusing an import

Write a class `ForbiddingFinder`, a finder which receives a set of forbidden module names when it's created and raises an `ImportError` on the import of one of them or of one of their submodules (`ftplib`, but also `wsgiref.simple_server` if `wsgiref` is forbidden). For any other name, it hands over to the next finder. Insert one at the head of `sys.meta_path`, try to import a forbidden module, then remove it from `sys.meta_path`, so that the rest of the session imports normally.

With `ForbiddingFinder({"ftplib", "wsgiref"})`, this code:

```python
for name in ["ftplib", "wsgiref.simple_server", "csv", "json"]:
  try:
    importlib.import_module(name)
    print(f"{name} imported")
  except ImportError as error:
    print(error)
```

must print:

```text
import of ftplib forbidden
import of wsgiref forbidden
csv imported
json imported
```

Why does `json` get imported even if you add it to the forbidden names?

In [ ]:
# Your code here

#### Additional hints

A finder that raises an `ImportError` in `find_spec` stops the import: the next finders aren't queried. `import wsgiref.simple_server` first imports the `wsgiref` package: that's the name `find_spec` receives, hence the message. But if the package is already loaded, only `"wsgiref.simple_server"` reaches it: `name.partition(".")[0]` gives `"wsgiref"`. This finder loads nothing: it doesn't need to be a `Loader`. Remove it with `sys.meta_path.remove(finder)` in a `finally`, so that an exception in the loop doesn't leave it in place.

```python
class ForbiddingFinder(importlib.abc.MetaPathFinder):
  def __init__(self, forbidden):
    self.forbidden = forbidden

  def find_spec(self, name, path, target=None):
    if ...:
      raise ImportError(f"import of {name} forbidden")
    return None


finder = ForbiddingFinder({"ftplib", "wsgiref"})
sys.meta_path.insert(0, finder)
try:
  ...
finally:
  sys.meta_path.remove(finder)
```

#### Solution

In [ ]:
class ForbiddingFinder(importlib.abc.MetaPathFinder):
  def __init__(self, forbidden):
    self.forbidden = forbidden

  def find_spec(self, name, path, target=None):
    if name.partition(".")[0] in self.forbidden:
      raise ImportError(f"import of {name} forbidden")
    return None


finder = ForbiddingFinder({"ftplib", "wsgiref", "json"})
sys.meta_path.insert(0, finder)
try:
  for name in ["ftplib", "wsgiref.simple_server", "csv", "json"]:
    try:
      importlib.import_module(name)
      print(f"{name} imported")
    except ImportError as error:
      print(error)
finally:
  sys.meta_path.remove(finder)

`json` is already in `sys.modules` (the notebook imported it for `ExportJson`, and Jupyter before it): `import` finds it in the cache, without querying any finder. A hook only sees the imports of a module not loaded yet, and code can remove the hook from `sys.meta_path`: it's not a sandbox. To run code you don't trust, you have to isolate it outside Python (a container, a process with reduced privileges).